# `ptof_obs_liveness_detection`

## What this notebook does
Detects liveness and data-quality gaps across the pipeline:
1. **Capability silence** (WARN) — a registered capability hasn't produced output in >grace_hours
2. **Capability silence ceiling** (CRITICAL, added 2026-09-21 as WARN; promoted CRITICAL and
   tightened 168h→120h 2026-09-21) — backstop for capabilities excluded from #1 because
   `silence_grace_hours IS NULL` (irregular cadence); fires only past a much longer fixed
   ceiling (`silence_ceiling_hours`)
3. **Shift context missing** (WARN) — blank shift_date/shift_type/batch_nbr in output records
4. **ETL pipeline health** (CRITICAL) — an upstream ETL task failed, meaning the agent is
   running on stale source data even though it's still producing outputs
5. **ETL table staleness** (CRITICAL, added 2026-09-21) — per-table companion to
   `etl_pipeline_staleness` (the global scalar check in `ptof_obs_alert.ipynb`): flags any single
   ETL source table that hasn't completed a run in >60 min, independent of whether the rest of
   the fleet is still running. Closes a confirmed masking gap — during the 2026-09-19/20
   weekend outage, 14 of 19 tables stalled for ~25h while the other 5 kept the global
   `max(run_timestamp)` fresh, so the global check never fired. Runs alongside the global check,
   not instead of it — see HANDOFF.md for the supplement-vs-replace reasoning (signed off
   2026-09-21).
6. **Write lag anomalies** (WARN, added 2026-09-18) — write_lag_s (ingestion_ts - called_at) for
   a capability/scheduler_run has degraded beyond `greatest(MAD-based bound, 300s)` (floor added
   2026-09-21 — see below)
7. **ETL run slow** (CRITICAL, added 2026-09-18 as WARN; promoted 2026-09-21) — an ETL task's
   duration_seconds has degraded beyond its own MAD-based historical bound, without outright
   failing; rolled up to `(task_name, window_start)` (grain changed 2026-09-21 — see below)

## Position in the pipeline
- **Job:** `obs_fresh_scan`, task `02_latency_detection` — runs after `01_bronze_projections`,
  before `06_alert`. (Items 6-7 are the first genuinely latency-related logic in this task —
  previously the task name didn't match its contents.)
- **Upstream:** reads `v_llm_bronze`, `v_etl_bronze` (built by `ptof_obs_bronze_projection`),
  `capability_registry` (human-curated by `ptof_obs_setup_seed`), and (added 2026-09-18)
  `write_lag_baseline` / `etl_duration_baseline` (built by `ptof_obs_nightly_baseline`).
- **Downstream:** `ptof_obs_alert.ipynb` reads `etl_pipeline_health`, `etl_table_staleness`,
  `etl_run_slow`, and `capability_silence_ceiling` (all CRITICAL) directly from tables via
  `INCIDENT_SOURCES`, and checks `capability_silence`, `shift_context_missing`,
  `write_lag_anomalies` (all WARN) via direct queries.

## Tables/views touched
- **Reads:** `v_llm_bronze`, `v_etl_bronze`, `capability_registry`, `write_lag_baseline`,
  `etl_duration_baseline`
- **Writes:** `capability_silence`, `capability_silence_ceiling`, `shift_context_missing`,
  `etl_pipeline_health`, `etl_table_staleness`, `write_lag_anomalies`, `etl_run_slow`

## Capability silence ceiling (added 2026-09-21 as WARN item #2; promoted CRITICAL 2026-09-21)
Backstop for any capability with `silence_grace_hours IS NULL` in `capability_registry`
(currently just `sev2-insights`) — those capabilities are structurally excluded from
`capability_silence` above, so before this addition there was no detection path at any length
if one went permanently dark. New `silence_ceiling_hours` column on `capability_registry`
(NULL for every capability except `sev2-insights`).

**Promoted WARN → CRITICAL and tightened 168h → 120h (2026-09-21, FP/FN bias review follow-up,
signed off: "ok lets do it").** Queried against `sev2-insights`'s full history (989 gaps): worst
gap ever observed 102.5h, p99 12.2h, zero gaps ever exceeded 120h or 168h. 120h was chosen over
the original 168h because it still clears the worst historical gap with room (~17%/17.5h margin,
same 0/989 empirical false-positive rate as 168h) while cutting the false-negative exposure
window by 29% — tightening to the evidence floor rather than leaving slack unused. Caveat
carried into `threshold_basis`: this bounds only the *known* failure shape (a 120h+ dark
period) on a single capability's thin history (~572 rows) — it doesn't guarantee no false
negative below that ceiling, doesn't account for future cadence drift, and (like `etl_run_slow`
before its own promotion) hasn't yet fired on a real event, so it should be revisited once it
has some live-fire history.

## ETL table staleness (added 2026-09-21, CRITICAL)
Per-table companion to the existing global `etl_pipeline_staleness` scalar check. Ships as
CRITICAL from day one (not staged as provisional/WARN like the 2026-09-18 MAD-based detectors)
because it reuses the already-proven simple-interval pattern (`minutes_since_last_run > N`)
rather than introducing a new unvalidated statistical model. Grouped directly off
`v_etl_bronze` — no `capability_registry`-style seed list needed, so coverage can't silently
drift if a table is added or removed upstream. Keyed on a constant `sha2(table_or_view, 256)`
(not table + window) so an ongoing stall is one incident whose `detection_count` climbs and
whose `resolved_at` auto-clears the moment the table resumes — same lifecycle as
`pipeline_heartbeat`/`etl_pipeline_staleness`, not the hourly-rollup pattern used by
`write_lag_anomalies`/`etl_run_slow`.

## Write lag anomalies / ETL run slow (added 2026-09-18)
Both were introduced WARN-tier and provisional: `ptof_obs_alert.ipynb` computed and printed
them to the job log, never persisted to `obs_incidents` and never posted to Teams — same
structural guarantee as the existing `capability_silence`/`shift_context_missing` WARNs. This
was deliberate: these are newly-introduced MAD-based thresholds (see `threshold_basis`,
`status='provisional'`) that hadn't been validated against real incident history yet, so they
ran in observe-only mode until proven not to be noisy. Each also rolls row-level anomalies up
into an hourly bin requiring >=3 occurrences before it's written at all, so isolated blips never
even reach the findings table.
- `write_lag_anomalies` catches "still writing, but slower than usual" — distinct from
  `capability_silence` ("stopped writing entirely"). **300s floor added 2026-09-21 (item #3,
  signed off):** the MAD baseline is degenerate (median=MAD=0 for every capability×scheduler_run,
  since write_lag_s is 0 for almost all prod rows), so the raw threshold was effectively "> 0".
  `greatest(upper_bound_s, 300)` ensures only genuinely extreme lag (5+ min) can trip this
  regardless of the useless baseline. Still WARN — the always-zero write_lag_s instrumentation
  question is on hold (2026-09-21, FP/FN bias review priority 3), so promotion isn't being
  considered until that's resolved.
- `etl_run_slow` catches "runs completing, but slower than usual" — distinct from
  `etl_pipeline_health` (outright failure) and `etl_pipeline_staleness` (no run at all).
  **Grain changed 2026-09-21 (item #4, signed off):** rolled up from
  `(table_or_view, task_name, window_start)` to `(task_name, window_start)` with
  `affected_tables`/`affected_table_count` in the payload, because all tables under one
  task_name move together — the old per-table grain produced 14-19 near-duplicate rows per
  real event. **Promoted WARN -> CRITICAL 2026-09-21 (FP/FN bias review priority 2, signed
  off):** 5 real, correlated, multi-table slowdown events were observed in ~1 week while this
  was log-only and invisible to a human — a proven-real signal, not a newly-introduced
  unvalidated one. Now persists to `obs_incidents` and posts to Teams like the CRITICAL
  detectors above.
Neither is a substitute for true per-call inference-latency detection (`latency_ms` on
`ptof_primary__ai_llm_audit_log`), which is blocked — that table has 0 rows in prod.

## Dropped detectors (prod migration 2026-09-10)
- `latency_anomalies` / `latency_anomaly_findings` — no `latency_ms` in prod
- `credential_fastfail_daily` — dev-specific model doesn't exist in prod
- `write_lag_daily` — depends on `latency_ms` for ingest-only computation
- `latency_failures` — depends on `success`, `error_class` (not in prod)
- `capability_health` / `capability_error_rate_alert` / `capability_error_rate_findings` — depends on `success`
- `prompt_size_drift` — no `user_prompt_chars` or `capability_latency_baseline` in prod

In [ ]:
%sql
-- capability_silence — WARN-tier liveness check: has each registered capability produced output
-- within its silence_grace_hours window? Joins capability_registry (active, non-null grace) against
-- v_llm_bronze (where output_type is aliased as capability, generated_at as called_at).
-- Prod capabilities: saa-display (2h), situational-awareness (2h), summary (36h).
-- sev2-insights has silence_grace_hours = NULL (irregular cadence) and is excluded by the WHERE.
CREATE OR REPLACE TABLE mq_gmdf_dev.oil_obs.capability_silence AS
SELECT
    r.capability, r.expected_min_daily, r.silence_grace_hours, r.owner,
    count(b.id)      AS calls_last_7d,
    max(b.called_at) AS last_call_at,
    round((unix_timestamp(current_timestamp()) - unix_timestamp(max(b.called_at)))/3600.0, 1)
                     AS hours_since_last_call
FROM mq_gmdf_dev.oil_obs.capability_registry r
LEFT JOIN mq_gmdf_dev.oil_obs.v_llm_bronze b
       ON b.capability = r.capability
      AND b.called_at >= current_timestamp() - INTERVAL 7 DAYS
WHERE r.active = true AND r.silence_grace_hours IS NOT NULL
GROUP BY 1, 2, 3, 4
HAVING max(b.called_at) IS NULL
    OR unix_timestamp(current_timestamp()) - unix_timestamp(max(b.called_at))
       > r.silence_grace_hours * 3600;

In [ ]:
%sql
-- capability_silence_ceiling (added 2026-09-21, item #2; promoted WARN -> CRITICAL and
-- tightened 168h -> 120h 2026-09-21, FP/FN bias review follow-up, signed off) -- backstop for
-- capabilities structurally EXCLUDED from capability_silence above because silence_grace_hours
-- IS NULL (irregular cadence). Confirmed gap (see HANDOFF.md "Detector value audit", #8):
-- sev2-insights has silence_grace_hours = NULL and had gone silent 64h+ with literally no
-- detection path at any length. This check does not replace the grace model -- it's a much
-- longer, fixed ceiling that only exists to catch "this capability has gone permanently dark,"
-- not routine irregular gaps (historical p95 0.2h, max 102.5h for sev2-insights).
--
-- silence_ceiling_hours is a separate column on capability_registry from silence_grace_hours
-- (NULL for every prod capability except sev2-insights, which is 120 -- tightened from 168 on
-- 2026-09-21: still 0/989 empirical false positives against sev2-insights's full gap history
-- (worst gap ever 102.5h), but cuts the false-negative exposure window vs. the original 168h).
-- CRITICAL as of 2026-09-21 -- promoted out of WARN because WARN-forever was itself judged a
-- false-negative risk (a real permanent-dark event would have gone unnotified indefinitely).
-- Caveat carried in threshold_basis: this has not yet fired on a real event, unlike etl_run_slow
-- before its own promotion -- revisit once it has some live-fire history.
--
-- finding_signature added 2026-09-21 alongside the CRITICAL promotion: a constant
-- sha2(capability, 256), same pattern as etl_table_staleness -- an ongoing silence is one
-- incident whose detection_count climbs and whose resolved_at auto-clears the moment the
-- capability produces output again, rather than a fresh row every run. Needed now that this
-- feeds ptof_obs_alert.ipynb's table-backed INCIDENT_SOURCES MERGE loop.
CREATE OR REPLACE TABLE mq_gmdf_dev.oil_obs.capability_silence_ceiling AS
SELECT
    r.capability, r.silence_ceiling_hours, r.owner,
    max(b.called_at) AS last_call_at,
    round((unix_timestamp(current_timestamp()) - unix_timestamp(max(b.called_at)))/3600.0, 1)
                     AS hours_since_last_call,
    sha2(r.capability, 256) AS finding_signature,
    current_timestamp() AS detected_at
FROM mq_gmdf_dev.oil_obs.capability_registry r
LEFT JOIN mq_gmdf_dev.oil_obs.v_llm_bronze b
       ON b.capability = r.capability
WHERE r.active = true
  AND r.silence_grace_hours IS NULL
  AND r.silence_ceiling_hours IS NOT NULL
GROUP BY r.capability, r.silence_ceiling_hours, r.owner
HAVING max(b.called_at) IS NULL
    OR unix_timestamp(current_timestamp()) - unix_timestamp(max(b.called_at))
       > r.silence_ceiling_hours * 3600;

In [ ]:
%sql
-- shift_context_missing — WARN-tier data-quality check: detects output records where shift
-- context fields (shift_date, shift_type, batch_nbr) are blank or null. These fields enable
-- per-shift and per-batch slicing and AI-to-ISH correlation. Reads v_llm_bronze (where
-- output_type is aliased as capability, generated_at as called_at). Scoped to active
-- capabilities via capability_registry inner join.
CREATE OR REPLACE TABLE mq_gmdf_dev.oil_obs.shift_context_missing AS
SELECT
    b.capability,
    count(*)                                                 AS total_calls,
    count_if(coalesce(b.shift_type, '') = '')                AS blank_shift_type,
    count_if(coalesce(cast(b.batch_nbr AS STRING), '') = '') AS blank_batch_nbr,
    count_if(b.shift_date IS NULL)                           AS null_shift_date,
    max(b.called_at)                                         AS last_seen,
    current_timestamp()                                      AS detected_at
FROM mq_gmdf_dev.oil_obs.v_llm_bronze b
JOIN mq_gmdf_dev.oil_obs.capability_registry r
  ON r.capability = b.capability AND r.active = true
WHERE b.called_at >= current_timestamp() - INTERVAL 7 DAYS
GROUP BY b.capability
HAVING count_if(coalesce(b.shift_type, '') = '') > 0
    OR count_if(coalesce(cast(b.batch_nbr AS STRING), '') = '') > 0
    OR count_if(b.shift_date IS NULL) > 0;

In [ ]:
%sql
-- etl_pipeline_health — CRITICAL detector: captures any ETL task failure in the last 24 hours.
-- The upstream ETL refreshes ~19 source tables every 10-15 min. When a task fails, the SAA
-- agent continues producing outputs using stale data — capability_silence and pipeline_heartbeat
-- won't fire because the agent is still generating, making this the only early warning.
-- finding_signature keyed on (table_or_view, run_id) so each distinct failure is one incident.
-- Reads v_etl_bronze (pass-through view over mq_gmdf_dp_prd.oil.ptof_etl_pipeline_audit).
CREATE OR REPLACE TABLE mq_gmdf_dev.oil_obs.etl_pipeline_health AS
SELECT
    run_id,
    run_timestamp,
    table_or_view,
    status,
    error_message,
    duration_seconds,
    sha2(concat_ws('|', table_or_view, run_id), 256) AS finding_signature,
    current_timestamp() AS detected_at
FROM mq_gmdf_dev.oil_obs.v_etl_bronze
WHERE status = 'failure'
  AND run_timestamp >= current_timestamp() - INTERVAL 24 HOURS;

In [ ]:
%sql
-- etl_table_staleness (added 2026-09-21, CRITICAL from day one) -- per-table companion to the
-- global etl_pipeline_staleness scalar check in ptof_obs_alert.ipynb. That global check takes
-- max(run_timestamp) across all 19 ETL source tables combined, which the 2026-09-19/20 weekend
-- incident showed can mask a partial stall: 14 of 19 tables went silent for ~25h while the
-- other 5 kept running on schedule, so the global max never went stale and nothing fired. This
-- check GROUPs BY table_or_view directly off v_etl_bronze (no capability_registry-style seed
-- list, so coverage can't drift if a table is added/removed upstream) and flags any single
-- table whose most recent run is more than 60 minutes old -- real fleet-wide cadence is
-- consistently ~12-16 min, so 60 min is a 4-6x margin above normal jitter, not a per-cycle
-- trigger.
--
-- Keyed on a constant sha2(table_or_view, 256) (not table + window), same lifecycle pattern as
-- pipeline_heartbeat/etl_pipeline_staleness: an ongoing stall is one incident whose
-- detection_count climbs in ptof_obs_alert.ipynb's MERGE loop, and whose resolved_at
-- auto-clears the moment the table resumes -- not a fresh row every run like the hourly-rollup
-- pattern used by write_lag_anomalies/etl_run_slow.
CREATE OR REPLACE TABLE mq_gmdf_dev.oil_obs.etl_table_staleness AS
SELECT
    table_or_view,
    max(run_timestamp) AS last_run_at,
    round((unix_timestamp(current_timestamp()) - unix_timestamp(max(run_timestamp)))/60.0, 1)
                        AS minutes_since_last_run,
    sha2(table_or_view, 256) AS finding_signature,
    current_timestamp()      AS detected_at
FROM mq_gmdf_dev.oil_obs.v_etl_bronze
GROUP BY table_or_view
HAVING unix_timestamp(current_timestamp()) - unix_timestamp(max(run_timestamp)) > 60 * 60;

In [ ]:
%sql
-- write_lag_anomalies (added 2026-09-18, WARN, provisional -- see threshold_basis) -- flags
-- v_llm_bronze rows whose write_lag_s exceeds that (capability, scheduler_run)'s MAD-based
-- upper_bound_s from write_lag_baseline. Distinct from capability_silence (which catches
-- "stopped writing entirely") -- this catches "still writing, but slower than its own history."
-- Row-level anomalies are logged here, but only rolled up into an hourly (capability,
-- window_start) bin with >=3 anomalous rows becomes a row in this table -- a single slow write
-- is noise; a cluster of 3+ in the same hour is a real degradation. WARN-tier: computed and
-- printed to the job log by ptof_obs_alert, never persisted to obs_incidents or posted to Teams.
--
-- 300s floor (added 2026-09-21, item #3, signed off): write_lag_baseline's median and MAD are
-- both 0 for every (capability, scheduler_run) -- write_lag_s is 0 for the overwhelming
-- majority of prod rows, so the MAD-based upper_bound_s collapses to 0.0000 and this check was
-- effectively "write_lag_s > 0", flagging any nonzero lag at all rather than a genuine
-- statistical anomaly. `greatest(base.upper_bound_s, 300)` puts a 5-minute floor under the
-- degenerate baseline so routine single-digit-second lag can't trip this -- only lag extreme
-- enough to matter regardless of the (currently useless) baseline will. 300s chosen over a
-- smaller placeholder specifically because the ask was to only catch extreme lag, not "more
-- than trivial."
CREATE OR REPLACE TABLE mq_gmdf_dev.oil_obs.write_lag_anomalies AS
WITH flagged AS (
    SELECT
        b.capability, b.scheduler_run, b.write_lag_s, b.called_at,
        window(b.called_at, '60 minutes').start AS window_start,
        greatest(base.upper_bound_s, 300) AS effective_upper_bound_s
    FROM mq_gmdf_dev.oil_obs.v_llm_bronze b
    JOIN mq_gmdf_dev.oil_obs.write_lag_baseline base
      ON base.capability = b.capability AND base.scheduler_run = b.scheduler_run
    WHERE b.called_at >= current_timestamp() - INTERVAL 24 HOURS
      AND b.write_lag_s > greatest(base.upper_bound_s, 300)
)
SELECT
    capability, scheduler_run, window_start,
    count(*)                     AS anomalous_count,
    max(write_lag_s)             AS max_write_lag_s,
    max(effective_upper_bound_s) AS upper_bound_s,
    sha2(concat_ws('|', capability, scheduler_run, cast(window_start AS STRING)), 256)
                                  AS finding_signature,
    current_timestamp()          AS detected_at
FROM flagged
GROUP BY capability, scheduler_run, window_start
HAVING count(*) >= 3;

In [ ]:
%sql
-- etl_run_slow (added 2026-09-18, WARN, provisional; promoted to CRITICAL 2026-09-21 -- see
-- threshold_basis) -- flags v_etl_bronze runs whose duration_seconds exceeds that
-- (table_or_view, task_name)'s MAD-based upper_bound_s from etl_duration_baseline. Distinct
-- from etl_pipeline_health (which catches outright failures) and etl_pipeline_staleness (which
-- catches "no run completed at all") -- this catches "runs are completing, but taking longer
-- than their own history," an early warning that can precede an actual failure or staleness
-- incident.
--
-- Grain (changed 2026-09-21, item #4, signed off): rolled up to (task_name, window_start),
-- NOT (table_or_view, task_name, window_start). The audit found all tables under one task_name
-- always move together -- the 4 real slowdown events in 7 days each hit 14-19 of 19 tables
-- simultaneously, so the old per-table grain produced 14-19 near-duplicate rows for a single
-- real event. affected_tables/affected_table_count preserve which tables were involved without
-- fragmenting one event into many rows. Row-level slow runs are still logged per-table, but the
-- >=3-in-an-hour rollup that decides whether anything gets written now groups by task_name.
--
-- Promoted WARN -> CRITICAL (2026-09-21, FP/FN bias review priority 2, signed off): 5 real,
-- correlated, multi-table slowdown events were observed in ~1 week while this was log-only and
-- invisible to a human -- a proven-real signal. ptof_obs_alert.ipynb now persists this to
-- obs_incidents and posts it to Teams via the standard CRITICAL lifecycle.
CREATE OR REPLACE TABLE mq_gmdf_dev.oil_obs.etl_run_slow AS
WITH flagged AS (
    SELECT
        e.table_or_view, e.task_name, e.duration_seconds, e.run_timestamp,
        window(e.run_timestamp, '60 minutes').start AS window_start,
        base.upper_bound_s
    FROM mq_gmdf_dev.oil_obs.v_etl_bronze e
    JOIN mq_gmdf_dev.oil_obs.etl_duration_baseline base
      ON base.table_or_view = e.table_or_view AND base.task_name = e.task_name
    WHERE e.run_timestamp >= current_timestamp() - INTERVAL 24 HOURS
      AND e.status = 'success'
      AND e.duration_seconds > base.upper_bound_s
)
SELECT
    task_name, window_start,
    count(*)                                       AS anomalous_count,
    count(distinct table_or_view)                  AS affected_table_count,
    concat_ws(', ', collect_set(table_or_view))     AS affected_tables,
    max(duration_seconds)                          AS max_duration_s,
    max(upper_bound_s)                             AS upper_bound_s,
    sha2(concat_ws('|', task_name, cast(window_start AS STRING)), 256)
                                                     AS finding_signature,
    current_timestamp()                            AS detected_at
FROM flagged
GROUP BY task_name, window_start
HAVING count(*) >= 3;